# Curating systems for SLiM simulations

**Useful commands**

Submitting many slurm scripts
* In system dir: ``for d in */; do (cd "$d" && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && sbatch gpu.sh); done)); done``

Submitting slurm scripts where no ``.dcd`` files exist
* In system dir: ``for d in */; do (cd "$d" && [ ! -f *.dcd ] && sbatch gpu.sh); done``
* In sampling dir: ``for d in */; do (cd "$d" && (for s in */; do (cd "$s" && [ ! -f *.dcd ] && sbatch gpu.sh); done)); done``

Manual SMOG run
* PDBAdjust: ``~/software/smog-2.5/bin/smog_adjustPDB -i $raw_pdb -o $adjusted_pdb -removewater -renumber -d``

In [4]:
import json
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import shutil

os.chdir('/home/fknudsen/calvados-slim')

from src import curation, utils

In [5]:
# Setting SMOG bin path, for SCM calculations
os.environ["SMOGBIN"] = os.path.expanduser("~/software/smog-2.5/bin")

In [6]:
# Loading curations
with open('data/curation/systems.json') as file:
    systems = json.load(file)
systems_dir = os.path.abspath('data/curation/systems')

# Loading configurations
with open('data/configurations/conditions.json') as file:
    conditions = json.load(file)
with open('data/configurations/sampling.json') as file:
    sampling = json.load(file)
with open('data/configurations/go_scans.json') as file:
    go_scans = json.load(file)
slurm_scripts_dir = 'data/configurations/slurm'

In [ ]:
# Curating input files for systems
for system in tqdm(systems):
    curation.System.from_json(system, dir=systems_dir)

100%|██████████| 8/8 [00:56<00:00,  7.07s/it]


In [8]:
# Setting modes to create runs for
conditions_mode = 'Default'
sampling_mode = 'Scan'
go_scan_mode = 'Thorough limited'

In [9]:
# Generating simulation setups
scan_dir = os.path.abspath(f"sim/{sampling_mode.lower()}")

# Using date as hash
hash = pd.Timestamp.now('utc').strftime("%Y%m%d")

# Generating Go potential k-range
k_min, k_max, k_step = go_scans[go_scan_mode]['Lower'], go_scans[go_scan_mode]['Upper'], go_scans[go_scan_mode]['Step']
ks = np.arange(k_min, k_max+k_step, k_step)

# Looping over systems
for system in systems:
    if 'sh3-prm' in system['ID']:
        continue
    system_dir = os.path.join(systems_dir, system['ID'])

    # Looping over Go k scan
    for k in ks:

        # Creating simulation directory
        sim_dir = os.path.join(scan_dir, system['ID'], f"k={k:.2f}")
        if os.path.exists(sim_dir):
            continue
        print(f"Creating '{sim_dir}'")
        os.makedirs(sim_dir)
        os.makedirs(os.path.join(sim_dir, 'logs'))

        curation.Simulation.from_jsons(
            system,
            system_dir,
            conditions[conditions_mode],
            sampling[sampling_mode],
            k, sim_dir
        )


Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.00'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.10'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.20'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.30'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.40'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.50'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.60'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.70'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.80'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=2.90'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=3.00'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=3.10'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=3.20'
Creating '/home/fknudsen/calvados-slim/sim/scan/klhdc2-ephb2/k=3.30'
Creating '/home/fknudsen/calvados-

In [ ]:
# Parsing a residue mapping in SMOG input PDB
residue_map = []
with open('/home/fknudsen/calvados-slim/.old/temp/adjusted.pdb', 'r') as f:
    for line in f:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            try:
                atom_serial = int(line[6:11].strip())
                res_name = line[17:20].strip()
                chain_id = line[21].strip()
                res_seq = int(line[22:26].strip())
                residue_map += [(chain_id, res_seq, atom_serial)]
            except ValueError:
                continue
residue_map = pd.DataFrame(residue_map, columns=['chainid', 'resno', 'atomid'])
residue_map['chainno'] = pd.factorize(residue_map['chainid'])[0] + 1
residue_map.set_index(['chainno','atomid'], inplace=True)

# Loading SMOG output
smog_contacts = pd.read_csv(
    f'/home/fknudsen/calvados-slim/.old/temp/smog.contacts',
    delimiter=' ',
    names=['chainno_1', 'atomid_1', 'chainno_2', 'atomid_2']
)

# Initialising empty contact map
chain_1_len = residue_map[residue_map['chainid'] == 'A']['resno'].nunique()
chain_2_len = residue_map[residue_map['chainid'] == 'B']['resno'].nunique()
contacts = np.zeros((chain_1_len, chain_2_len), dtype=bool)

# Looking for inter-chain AA contacts
for _, row in smog_contacts.iterrows():
    cid_1, rno_1 = residue_map.loc[(row['chainno_1'], row['atomid_1'])]
    cid_2, rno_2 = residue_map.loc[(row['chainno_2'], row['atomid_2'])]
    ridx_1 = rno_1 - 1
    ridx_2 = rno_2 - 1

    if (cid_1 == 'A') & (cid_2 == 'B'):
        contacts[ridx_1, ridx_2] = True
    elif (cid_1 == 'B') & (cid_2 == 'A'):
        contacts[ridx_2, ridx_1] = True